# EduPulse AI — Exploratory Data Analysis (Milestone 1)

**Project:** *EduPulse AI: Early-Warning Performance Risk Classifier & Socratic Coaching Platform*
**Dataset:** Open University Learning Analytics Dataset (OULAD), CC BY 4.0
**Milestone 1 goal:** understand and clean the data, then produce a leakage-free feature table.

This notebook is **executed**, not a template: every table and figure below was produced by running
the cells against the real files in `data/raw/`. It documents *why* the preprocessing pipeline makes
the choices it makes.

The heavy `studentVle.csv` (10.66 M rows, ~454 MB) is read in chunks; it is never loaded whole.

In [1]:
import warnings, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# Resolve the project root whether the notebook is run from /notebooks or the root
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW, PROC, REP = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports"
FIG = REP / "figures"; FIG.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))
from preprocess import RawSource, RAW_FILES
SRC = RawSource(RAW)

def read_raw(name, **kw):
    """Read one OULAD table, whether extracted or inside the archive."""
    with SRC.open(name) as fh:
        return pd.read_csv(fh, **kw)

def raw_chunks(name, **kw):
    """Chunked reader for the large VLE table."""
    with SRC.open(name) as fh:
        yield from pd.read_csv(fh, **kw)

print("project root:", ROOT)
print("raw data source:", SRC.describe())

[preprocess] reading OULAD directly from anonymisedData.zip (46.8 MB, no extraction needed)


project root: /home/user/EduPulse-AI
raw data source: archive anonymisedData.zip (read without extracting)


## 1. File inventory

OULAD ships seven CSVs at different grains. Getting the grain right is the whole game:
`studentInfo` is one row per enrolment, `studentAssessment` one row per submission,
`studentVle` one row per interaction record, and the metadata tables describe courses,
assessments and VLE resources.

In [2]:
import os, subprocess
rows = []
for name in RAW_FILES:
    if name == "studentVle.csv":
        n = sum(len(c) for c in raw_chunks(name, usecols=["date"], chunksize=2_000_000))
    else:
        n = len(read_raw(name, usecols=[0]))
    rows.append({"file": name, "size_mb": round(SRC.size(name)/1e6, 2),
                 "rows": n, "columns": len(read_raw(name, nrows=0).columns)})
inventory = pd.DataFrame(rows)
inventory

,file,size_mb,rows,columns
0,courses.csv,0.00,22,3
1,assessments.csv,0.01,206,6
2,studentInfo.csv,3.46,32593,12
3,studentRegistration.csv,1.11,32593,5
4,studentAssessment.csv,5.69,173912,5
5,vle.csv,0.26,6364,6
6,studentVle.csv,453.84,10655280,6


The 10.66 M-row VLE log dominates the data volume but carries only six columns; the small
tables carry the semantics. Note that `studentInfo` has 32,593 rows but only 28,785 distinct students —
**a student can appear in several presentations**, so `id_student` alone is not an enrolment key.

In [3]:
for name in ["courses.csv", "assessments.csv", "studentInfo.csv", "studentRegistration.csv",
             "studentAssessment.csv", "vle.csv"]:
    df = read_raw(name, nrows=5)
    print(f"\n=== {name} ===")
    print(df.dtypes.to_string())
    display(df.head(3))


=== courses.csv ===
code_module                   object
code_presentation             object
module_presentation_length     int64


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268



=== assessments.csv ===
code_module          object
code_presentation    object
id_assessment         int64
assessment_type      object
date                  int64
weight                int64


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10
1,AAA,2013J,1753,TMA,54,20
2,AAA,2013J,1754,TMA,117,20



=== studentInfo.csv ===
code_module             object
code_presentation       object
id_student               int64
gender                  object
region                  object
highest_education       object
imd_band                object
age_band                object
num_of_prev_attempts     int64
studied_credits          int64
disability              object
final_result            object


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn


... (2 outputs left out)


=== studentAssessment.csv ===
id_assessment     int64
id_student        int64
date_submitted    int64
is_banked         int64
score             int64


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72



=== vle.csv ===
id_site                int64
code_module           object
code_presentation     object
activity_type         object
week_from            float64
week_to              float64


,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,NaN,NaN
1,546712,AAA,2013J,oucontent,NaN,NaN
2,546998,AAA,2013J,resource,NaN,NaN


## 2. Missing values

Missingness in OULAD is *structural*, not random: a student who never withdraws has no
`date_unregistration`, and `imd_band` is simply not recorded for some students.
Treating these as zero would be a modelling error, so the pipeline keeps them as `NaN`
and adds explicit indicator columns where it matters.

In [4]:
miss = []
for name in ["courses.csv", "assessments.csv", "studentInfo.csv",
             "studentRegistration.csv", "studentAssessment.csv", "vle.csv"]:
    df = read_raw(name)
    for col, v in df.isna().sum().items():
        if v:
            miss.append({"file": name, "column": col, "missing": int(v),
                         "pct": round(100*v/len(df), 2)})
miss_df = pd.DataFrame(miss).sort_values("pct", ascending=False)
display(miss_df)

fig, ax = plt.subplots(figsize=(8, 3.2))
top = miss_df.head(8).iloc[::-1]
ax.barh(top["file"] + " :: " + top["column"], top["pct"], color="#c0504d")
ax.set_xlabel("% missing"); ax.set_title("Most-missing columns (small tables)")
fig.tight_layout(); fig.savefig(FIG / "01_missingness.png", bbox_inches="tight"); plt.show()

,file,column,missing,pct
6,vle.csv,week_to,5243,82.39
5,vle.csv,week_from,5243,82.39
3,studentRegistration.csv,date_unregistration,22521,69.10
0,assessments.csv,date,11,5.34
1,studentInfo.csv,imd_band,1111,3.41
2,studentRegistration.csv,date_registration,45,0.14
4,studentAssessment.csv,score,173,0.10


`date_unregistration` is missing for ~69% of enrolments — those students stayed to the end.
That is the *absence of an event*, not a missing measurement. `imd_band` is missing for
1,111 enrolments (3.4%) and is handled with a separate `imd_band_missing` flag rather than imputation.

## 3. Duplicates and candidate keys

Before joining anything, establish which column combinations are unique. A duplicated join key
silently multiplies rows — the classic way student-level features get corrupted.

In [5]:
checks = []
info = read_raw("studentInfo.csv"); reg = read_raw("studentRegistration.csv")
assess = read_raw("assessments.csv"); sa = read_raw("studentAssessment.csv")
vle = read_raw("vle.csv"); courses = read_raw("courses.csv")
K = ["code_module", "code_presentation", "id_student"]

def add(table, label, cols, df):
    checks.append({"table": table, "candidate key": label, "rows": len(df),
                   "duplicate rows": int(df.duplicated(cols).sum()),
                   "unique?": "YES" if not df.duplicated(cols).any() else "NO"})

add("studentInfo", "(module, presentation, student)", K, info)
add("studentRegistration", "(module, presentation, student)", K, reg)
add("assessments", "id_assessment", ["id_assessment"], assess)
add("assessments", "(module, presentation, id_assessment)", ["code_module","code_presentation","id_assessment"], assess)
add("studentAssessment", "(id_assessment, id_student)", ["id_assessment","id_student"], sa)
add("vle", "id_site", ["id_site"], vle)
add("courses", "(module, presentation)", ["code_module","code_presentation"], courses)
pd.DataFrame(checks)

,table,candidate key,rows,duplicate rows,unique?
0,studentInfo,"(module, presentation, student)",32593,0,YES
1,studentRegistration,"(module, presentation, student)",32593,0,YES
2,assessments,id_assessment,206,0,YES
3,assessments,"(module, presentation, id_assessment)",206,0,YES
4,studentAssessment,"(id_assessment, id_student)",173912,0,YES
5,vle,id_site,6364,0,YES
6,courses,"(module, presentation)",22,0,YES


Every join key above is unique, so the enrichment joins in the pipeline are one-to-many and
cannot inflate the enrolment count. The pipeline still passes `validate="many_to_one"` on every merge
and asserts that row counts are preserved.

`studentVle` is the exception — see the next section.

## 4. The VLE log: 10.66 M rows, read in chunks

`sum_click` is a **count of recorded interactions**, not time spent studying. The intended grain is
one row per (student, resource, day) — but the real file violates it, so this must be measured rather
than assumed.

In [6]:
n_rows = 0; n_dup = 0; neg = 0; zero = 0
dmin, dmax = 10**9, -10**9
students, sites = set(), set()
weekly = {}
key_hash_parts, click_parts = [], []

for ch in raw_chunks("studentVle.csv", chunksize=2_000_000,
                      dtype={"id_student":"int32","id_site":"int32","date":"int16","sum_click":"int32"}):
    n_rows += len(ch)
    n_dup  += int(ch.duplicated().sum())
    neg    += int((ch.sum_click < 0).sum()); zero += int((ch.sum_click == 0).sum())
    dmin, dmax = min(dmin, int(ch.date.min())), max(dmax, int(ch.date.max()))
    students.update(ch.id_student.unique().tolist()); sites.update(ch.id_site.unique().tolist())
    wk = ch.groupby((ch.date // 7).astype(int)).sum_click.sum()
    for k, v in wk.items():
        weekly[int(k)] = weekly.get(int(k), 0) + int(v)
    key_hash_parts.append(pd.util.hash_pandas_object(ch[["code_module","code_presentation","id_student","id_site","date"]], index=False).to_numpy())
    click_parts.append(ch.sum_click.to_numpy())

print(f"rows            : {n_rows:,}")
print(f"duplicate rows  : {n_dup:,} ({100*n_dup/n_rows:.2f}%)")
print(f"negative clicks : {neg:,} | zero clicks: {zero:,}")
print(f"date range      : {dmin} .. {dmax} course days")
print(f"distinct students {len(students):,} | distinct sites {len(sites):,}")

kh = np.concatenate(key_hash_parts); ck = np.concatenate(click_parts)
_, counts = np.unique(kh, return_counts=True)
print(f"distinct (module,pres,student,site,date) keys: {len(counts):,}")
print("key multiplicity:", {k: int(v) for k, v in
      {"1 row": (counts==1).sum(), "2 rows": (counts==2).sum(), "3-5 rows": ((counts>=3)&(counts<=5)).sum(),
       ">5 rows": (counts>5).sum()}.items()})

rows            : 10,655,280
duplicate rows  : 787,168 (7.39%)
negative clicks : 0 | zero clicks: 0
date range      : -25 .. 269 course days
distinct students 26,074 | distinct sites 6,268


distinct (module,pres,student,site,date) keys: 8,459,320
key multiplicity: {'1 row': 6844815, '2 rows': 1311841, '3-5 rows': 250886, '>5 rows': 51778}


**Finding:** the VLE file is *not* unique on its documented key: the real run above reports
8,459,320 distinct (module, presentation, student, site, date) keys for 10,655,280 rows (7.39% exact
duplicate rows), and roughly 1.6 M keys carry more than one row. Since `sum_click` is an additive count of interactions,
the pipeline aggregates with `sum()` per (student, site, day) instead of dropping rows — no recorded
interaction is discarded.

The date range also runs **before day 0** (down to −25) and **past the course end** (up to 269):
students browse early, and the log keeps running afterwards. Both matter for how a cutoff is defined.

In [7]:
weeks = pd.Series(weekly).sort_index()
weeks = weeks[(weeks.index >= -4) & (weeks.index <= 30)]
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(weeks.index, weeks.values, color=np.where(weeks.index < 0, "#c0504d", "#4f81bd"))
ax.axvline(28/7, color="black", ls="--", lw=1.5)   # day 28 == week 4 on this axis
ax.text(28/7 + 0.4, weeks.max()*0.92, "day-28 cutoff (week 4)", fontsize=9)
ax.set_xlabel("course week (week 0 = first 7 days; negative = before the course)")
ax.set_ylabel("recorded clicks")
ax.set_title("Recorded VLE clicks per course week, all 22 presentations")
fig.tight_layout(); fig.savefig(FIG / "02_clicks_per_week.png", bbox_inches="tight"); plt.show()

print(f"clicks before day 0 : {weekly.get(-1,0)+weekly.get(-2,0)+weekly.get(-3,0)+weekly.get(-4,0):,}")
print(f"clicks days 0-27    : {sum(v for k,v in weekly.items() if 0 <= k <= 3):,}")

clicks before day 0 : 2,147,947
clicks days 0-27    : 7,781,403


Activity peaks in the opening weeks (onboarding) and the red pre-course bars are substantial:
a model that ignores pre-course activity throws away real signal. After the peak, engagement decays —
which is exactly what an early-warning system should exploit.

## 5. Assessment semantics — the trap in this dataset

Two things about `assessments.csv` break naive assumptions:

1. **`weight` is not a percentage of the final mark.** Summing it per presentation gives 100–300.
2. **11 assessments have no due date at all**, so they cannot be placed before or after a cutoff.

In [8]:
wsum = assess.groupby(["code_module","code_presentation"])["weight"].sum()
print("weight summed per presentation:")
print(wsum.describe().round(1).to_string())
print("\nweight summed per (presentation, assessment_type):")
print(assess.groupby(["code_module","code_presentation","assessment_type"])["weight"].sum()
        .unstack().fillna("—").to_string())
print(f"\nassessments with no due date: {assess.date.isna().sum()} ->")
print(assess[assess.date.isna()][["code_module","code_presentation","id_assessment","assessment_type","weight"]].to_string(index=False))

weight summed per presentation:
count     22.0
mean     195.5
std       48.6
min      100.0
25%      200.0
50%      200.0
75%      200.0
max      300.0

weight summed per (presentation, assessment_type):
assessment_type                 CMA   Exam    TMA
code_module code_presentation                    
AAA         2013J                 —  100.0  100.0
            2014J                 —  100.0  100.0
BBB         2013B               5.0  100.0   95.0
            2013J               5.0  100.0   95.0
            2014B               5.0  100.0   95.0
            2014J                 —  100.0  100.0
CCC         2014B              25.0  200.0   75.0
            2014J              25.0  200.0   75.0
DDD         2013B              25.0  100.0   75.0
            2013J                 —  100.0  100.0
            2014B                 —  100.0  100.0
            2014J                 —  100.0  100.0
EEE         2013J                 —  100.0  100.0
            2014B                 —  100.0  10

The per-type totals (TMA/Exam ≈ 100, CMA often 0/5/25, CCC exams 200) show that `weight` is a
**within-type** importance. The pipeline therefore normalises weight inside (presentation, type) before
computing `weighted_mean_score`; using the raw numbers would let CCC's exams count double.

The 11 undated assessments are all Exams. Do their submissions really fall at the end of the course?

In [9]:
m = sa.merge(assess[["id_assessment","date","assessment_type"]], on="id_assessment", how="left")
print("submission days for UNDATED assessments:")
print(m.loc[m.date.isna(), "date_submitted"].describe().round(1).to_string())
print("\ndue days for DATED exams:", sorted(assess.loc[assess.assessment_type=="Exam","date"].dropna().astype(int).unique()))
print("\ndated assessments due by day 28:", int((assess.date <= 28).sum()), "of", len(assess))

m = m[m.date.notna() & (m.date <= 28) & (m.date_submitted <= 28)].copy()
m["delay"] = m.date_submitted - m.date
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(m.delay.clip(-30, 30), bins=60, color="#4f81bd")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("submission delay in days (negative = handed in early)")
ax.set_ylabel("submissions"); ax.set_title("Submission timing for assessments due by day 28")
fig.tight_layout(); fig.savefig(FIG / "03_submission_delay.png", bbox_inches="tight"); plt.show()
print(f"median delay: {m.delay.median():.0f} days | {(m.delay<0).mean()*100:.1f}% submitted early")

submission days for UNDATED assessments:
count    2865.0
mean      240.5
std         5.7
min       230.0
25%       236.0
50%       243.0
75%       244.0
max       285.0

due days for DATED exams: [np.int64(222), np.int64(227), np.int64(228), np.int64(229), np.int64(235), np.int64(236), np.int64(240), np.int64(241), np.int64(261)]

dated assessments due by day 28: 17 of 206
median delay: -1 days | 56.7% submitted early


The undated exams are submitted around day 230–285 — the end of the course. Dropping them costs
**no early-warning signal**; it only removes rows that could never be observed before an early cutoff.

Submission delay is a real behavioural signal: about half of the submissions arrive before the due date,
and the tail of late submissions is where struggling students show up.

## 6. Who is at risk? The outcome distribution

OULAD's `final_result` has four values. The project's target collapses {Fail, Withdrawn} into
`at_risk = 1`. This binary label is **derived here**, not supplied by the dataset.

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
vc = info.final_result.value_counts().reindex(["Pass","Fail","Withdrawn","Distinction"])
axes[0].bar(vc.index, vc.values, color=["#4f81bd","#c0504d","#9bbb59","#8064a2"])
axes[0].set_title("final_result across all 32,593 enrolments"); axes[0].set_ylabel("enrolments")
for i, v in enumerate(vc.values): axes[0].text(i, v+200, f"{v:,}", ha="center", fontsize=9)

info["risk"] = info.final_result.isin(["Fail","Withdrawn"]).astype(int)
byp = info.groupby(["code_module","code_presentation"])["risk"].mean().mul(100).sort_values()
axes[1].barh([f"{a}-{b}" for a,b in byp.index], byp.values, color="#c0504d")
axes[1].set_title("at-risk rate by presentation (%)"); axes[1].tick_params(labelsize=7)
fig.tight_layout(); fig.savefig(FIG / "04_outcomes.png", bbox_inches="tight"); plt.show()

print(info.final_result.value_counts(normalize=True).mul(100).round(2).to_string())
print(f"\noverall at-risk share: {info.risk.mean()*100:.1f}%")
print("risk varies by presentation from", round(byp.min(),1), "to", round(byp.max(),1), "%")

final_result
Pass           37.93
Withdrawn      31.16
Fail           21.64
Distinction     9.28

overall at-risk share: 52.8%
risk varies by presentation from 27.4 to 65.8 %


Two consequences for the November model:

* **Class balance:** at-risk is ~53% of all enrolments (44% after excluding pre-cutoff withdrawals),
  not a rare-event problem — plain accuracy will be misleading, so precision/recall/AUC will be needed.
* **Presentation effects:** CCC fails far more often than AAA. Since `code_module` is not a legitimate
  predictor (it is an identifier), the model must find the signal in behaviour and scores, and evaluation
  should be done on held-out *presentations*, not random rows.

## 7. Choosing the prediction cutoff

The cutoff decides when the prediction is made. Too early and there are no assessment results;
too late and there is no time to intervene. This is measured, not guessed.

In [11]:
KEY = ["code_module","code_presentation","id_student"]
cut_rows = []
for c in [7, 14, 21, 28, 42, 56]:
    due = assess[(assess.date.notna()) & (assess.date <= c)]
    obs = sa.merge(due[["id_assessment","code_module","code_presentation"]], on="id_assessment", how="inner")
    obs = obs[obs.date_submitted <= c]
    per = obs[obs.score.notna()].groupby(KEY).size()
    j = info[KEY].merge(per.rename("n").reset_index(), on=KEY, how="left").fillna({"n":0})
    cut_rows.append({"cutoff_day": c,
                     "presentations_with_a_due_assessment": due[["code_module","code_presentation"]].drop_duplicates().shape[0],
                     "pct_enrolments_with_a_score": round(100*(j.n>=1).mean(),1),
                     "mean_assessments_graded": round(j.n.mean(),2)})
cut_df = pd.DataFrame(cut_rows); display(cut_df)

fig, ax1 = plt.subplots(figsize=(8, 3.3))
ax1.plot(cut_df.cutoff_day, cut_df.pct_enrolments_with_a_score, "o-", color="#4f81bd", label="% enrolments with a score")
ax1.set_xlabel("cutoff (course day)"); ax1.set_ylabel("% with a graded assessment", color="#4f81bd")
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(cut_df.cutoff_day, cut_df.presentations_with_a_due_assessment, "s--", color="#c0504d", label="presentations with an assessment due")
ax2.set_ylabel("presentations (of 22)", color="#c0504d")
ax1.axvline(28, color="black", ls=":"); ax1.set_title("Assessment signal available at each candidate cutoff")
fig.tight_layout(); fig.savefig(FIG / "05_cutoff_tradeoff.png", bbox_inches="tight"); plt.show()

,cutoff_day,presentations_with_a_due_assessment,pct_enrolments_with_a_score,mean_assessments_graded
0,7,0,0.0,0.00
1,14,1,3.3,0.03
2,21,11,42.3,0.42
3,28,16,62.8,0.66
4,42,19,71.6,0.88
5,56,19,72.3,1.38


**Chosen: day 28.** At day 28 (≈10% into a ~270-day presentation) 16 of 22 presentations already have
a due assessment and ~63% of enrolments have at least one graded result, while ~90% of the course is still
ahead. Waiting until day 56 would grade ~72% of enrolments and give a much stronger model, but leaves only
three quarters of the course to intervene — and still leaves GGG without any assessment.

The trade-off is why `--cutoff` is a parameter: the November model can be trained at 28 and 56 days and
compared on identical code.

## 8. What the processed feature table looks like

The pipeline output is one row per **enrolment at a cutoff** — `(code_module, code_presentation,
id_student, cutoff_day)` — with all predictors built only from information available by day 28.

In [12]:
feat = pd.read_csv(PROC / "student_features.csv")
print("shape:", feat.shape)
print("grain unique:", not feat.duplicated(["code_module","code_presentation","id_student","cutoff_day"]).any())
print("at-risk prevalence:", round(feat.target_at_risk.mean()*100, 1), "%")
print("rows with a graded result by cutoff:", round(feat.has_graded_assessment.mean()*100,1), "%")
print("rows with any VLE activity:", round(feat.has_vle_activity.mean()*100,1), "%")
display(feat[["code_module","code_presentation","n_assessments_due","n_assessments_graded",
              "mean_score","vle_clicks_total","vle_active_days","vle_clicks_last7d",
              "target_at_risk"]].head(8))

cols = ["mean_score","vle_clicks_total","vle_active_days","vle_distinct_sites",
        "assessment_submission_rate","vle_clicks_last7d"]
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5))
for ax, c in zip(axes.ravel(), cols):
    s = feat[c].dropna()
    ax.hist(np.log1p(s) if s.max() > 50 and c.startswith("vle") else s, bins=40, color="#4f81bd")
    ax.set_title(c, fontsize=10)
fig.suptitle("Distributions of key early-warning features (VLE features log1p-scaled)", y=1.02)
fig.tight_layout(); fig.savefig(FIG / "06_feature_distributions.png", bbox_inches="tight"); plt.show()

shape: (27522, 70)
grain unique: True
at-risk prevalence: 44.1 %
rows with a graded result by cutoff: 72.9 %
rows with any VLE activity: 96.7 %


,code_module,code_presentation,n_assessments_due,n_assessments_graded,mean_score,vle_clicks_total,vle_active_days,vle_clicks_last7d,target_at_risk
0,AAA,2013J,1,1,78.0,401.0,8.0,0.0,0
1,AAA,2013J,1,1,70.0,618.0,19.0,125.0,0
2,AAA,2013J,1,1,72.0,494.0,22.0,41.0,0
3,AAA,2013J,1,1,69.0,567.0,24.0,49.0,0
4,AAA,2013J,1,1,79.0,727.0,32.0,33.0,0
5,AAA,2013J,1,1,70.0,499.0,14.0,55.0,0
6,AAA,2013J,1,1,72.0,414.0,19.0,94.0,0
7,AAA,2013J,1,1,72.0,361.0,18.0,54.0,0


The heavy right tails in click counts are why the November model should use tree-based methods
(XGBoost handles skew and NaNs natively) rather than a linear model on raw counts.

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
for ax, c in zip(axes, ["vle_clicks_total","vle_active_days","mean_score"]):
    data = [np.log1p(feat.loc[feat.target_at_risk==g, c].dropna()) if c.startswith("vle")
            else feat.loc[feat.target_at_risk==g, c].dropna() for g in (0,1)]
    ax.boxplot(data, labels=["not at risk","at risk"], showfliers=False)
    ax.set_title(c, fontsize=10)
fig.suptitle("Descriptive only: early features by eventual outcome (no model fitted)", y=1.03)
fig.tight_layout(); fig.savefig(FIG / "07_features_by_outcome.png", bbox_inches="tight"); plt.show()

print(feat.groupby("target_at_risk")[["vle_clicks_total","vle_active_days","mean_score",
      "assessment_submission_rate"]].median().round(2).to_string())

/tmp/ipykernel_2863/2635489312.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=["not at risk","at risk"], showfliers=False)
/tmp/ipykernel_2863/2635489312.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=["not at risk","at risk"], showfliers=False)
/tmp/ipykernel_2863/2635489312.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=["not at risk","at risk"], showfliers=False)


                vle_clicks_total  vle_active_days  mean_score  assessment_submission_rate
target_at_risk                                                                           
0                          298.0             16.0        80.0                         1.0
1                          152.0              9.0        70.0                         1.0


Students who eventually fail or withdraw already show lower early engagement and lower early
scores by day 28 — the signal an early-warning model needs. This is a descriptive comparison, **not**
model validation; the classifier and its evaluation belong to Milestone 2.

The notebook prints the medians behind the figure: students not at risk record **298** clicks on **16**
active days with a median early score of **80**, versus **152** clicks on **9** days and **70** for
at-risk students. The boxes still overlap heavily in their upper/lower quartiles, so no single feature
separates the classes — which is the argument for a multivariate model rather than a threshold rule.

## 9. Limitations recorded for the next milestones

1. **No concept labels.** OULAD records whole-assessment results (TMA/CMA/Exam). There are no
   question-level items, concept tags or prerequisite graphs, so per-concept mastery
   (faculty Feature 2) cannot be computed from this dataset. `assessment_catalog.csv` is the join point
   where such labels would attach; `load_concept_labels()` validates a future
   `assessment_concepts.csv` automatically.
2. **Clicks ≠ study hours.** `sum_click` counts recorded platform interactions. No feature is described
   as study time.
3. **Some presentations have no assessment by day 28** (EEE and GGG), so their assessment features are
   structurally missing — preserved as `NaN`, never as zero.
4. **Withdrawal is ambiguous for 8 enrolments** whose `final_result` is Fail but who have an
   unregistration date; the pipeline excludes any enrolment unregistered by the cutoff, and documents it.
5. **Right-censoring:** the target uses the final outcome of the whole presentation, which is observed
   over the following ~240 days. It is a valid label for offline training but cannot be used online.

## 10. Bottom line

* Raw tables are understood, documented and joined without row multiplication.
* A leakage-free, reproducible feature table exists at a well-defined grain.
* The cutoff is a documented, tunable choice with a measured trade-off.
* The path to Milestone 2 (classification) and Milestone 3 (coaching) is prepared, including the
  explicit data gap that blocks concept-level mastery tracking.